逻辑分类识别垃圾邮件

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

In [5]:
# 1. 读取数据
data = pd.read_csv("../../../data/ml/SMSSpamCollection", sep="\t", names=["label", "message"])
print(data.head())
# ham 正常短信
# spam 垃圾短信

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [6]:
# 3. 数据处理


In [7]:
# 3. 特征与标签拆分
X = data["message"]
y = data["label"]
print(X.head())
print(y.head())

0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: str
0     ham
1     ham
2    spam
3     ham
4     ham
Name: label, dtype: str


In [8]:
# 4. 数据集拆分
# train:validation:test = 8:1:1
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.2, random_state=23)
X_test, X_validation, y_test, y_validation = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=23)
print(X_train.shape, X_test.shape, X_validation.shape)


(4457,) (557,) (558,)


In [10]:
# 5. 定义模型 训练
# 先学习文本向量化规则，再训练分类器
model = Pipeline([
    # TfidfVectorizer 会将文本分割，转化为数值
    ("tfidf", TfidfVectorizer()),
    ("lr", LogisticRegression(fit_intercept=True, max_iter=1000)),
])

model.fit(X_train, y_train)

print("coef_: ", model.named_steps["lr"].coef_)
print("intercept_: ", model.named_steps["lr"].intercept_)


coef_:  [[ 0.43370544  1.10102317 -0.01459682 ... -0.01179823 -0.02225105
  -0.01080269]]
intercept_:  [-2.48550421]


In [21]:
# 6. 评估验证集
y_pred = model.predict(X_validation)
pred_proba = model.predict_proba(X_validation)

print("accuracy_score: ", accuracy_score(y_validation, y_pred))
print("classification_report: ", classification_report(y_validation, y_pred))
print("confusion_matrix: ", confusion_matrix(y_validation, y_pred, labels=["ham", "spam"]))
spam_index = list(model.classes_).index("spam")
spam_proba = pred_proba[:, spam_index]
print("ROC-AUC:", roc_auc_score((y_validation == "spam").astype(int), spam_proba),)

print("pred_proba: ", pred_proba)


accuracy_score:  0.96415770609319
classification_report:                precision    recall  f1-score   support

         ham       0.96      1.00      0.98       476
        spam       0.98      0.77      0.86        82

    accuracy                           0.96       558
   macro avg       0.97      0.88      0.92       558
weighted avg       0.96      0.96      0.96       558

confusion_matrix:  [[475   1]
 [ 19  63]]
ROC-AUC: 0.9790684566509531
pred_proba:  [[0.92297598 0.07702402]
 [0.97644585 0.02355415]
 [0.95568782 0.04431218]
 ...
 [0.88140003 0.11859997]
 [0.97374534 0.02625466]
 [0.92623616 0.07376384]]


In [22]:
# 7. 预测一条新短信
new_message = [
    "Congratulations! You have won a free prize. Call now!"
]
print("预测结果：", model.predict(new_message))
print("类别概率：", model.predict_proba(new_message))

预测结果： ['spam']
类别概率： [[0.09859353 0.90140647]]


In [28]:
test2_X = [
    "free prize call now",
    "free money claim prize",
    "hello how are you",
    "are we meeting today",
    "where are from?",
    "This is the class and function reference of scikit-learn.",
]

test2_y = [
    "spam",
    "spam",
    "ham",
    "ham",
    "ham",
    "ham",
]

print("预测结果：", model.predict(test2_X))
print("预测结果：", model.predict(test2_X) == test2_y)
print("类别概率：", model.predict_proba(test2_X))

预测结果： ['spam' 'spam' 'ham' 'ham' 'ham' 'ham']
预测结果： [ True  True  True  True  True  True]
类别概率： [[0.04382027 0.95617973]
 [0.19010862 0.80989138]
 [0.97428666 0.02571334]
 [0.96074491 0.03925509]
 [0.83998922 0.16001078]
 [0.91649898 0.08350102]]
